In [ ]:
import pandas as pd 
from dotenv import load_dotenv
import os
from pathlib import Path
import xlrd
import numpy as np
import re

load_dotenv(override=True)

BANK_DIR = os.getenv("bank_dir")

def process_bank_file(file):
    df = pd.read_excel(file, engine = 'xlrd')
    
    muthoot_col = df.columns[df.columns.to_series().str.contains("muthoot", case=False)]
    dim_rec = {}
    if len(muthoot_col) > 0:
        df = df.rename(columns={muthoot_col[0]: "source_dim_col"})

        match = re.search(
            r"MUTHOOT FINANCE LTD\s+(\d+)-(.+?)/s+Bank Statement.*?Account No/s*[-:]?\s*(\d+)",
            muthoot_col[0], re.IGNORECASE
        )

        dim_rec = {
            # "source_dim_col" : muthoot_col[0],
            "branch_code" : match.group(1) if match else None,
            "branch_name"    : re.sub(r"[-]", " ", match.group(2)).strip() if match else None,
            "acc_no" : match.group(3) if match else None
        }
    opening_mask = (
        df.astype(str)
          .apply(lambda col: col.str.contains("opening", case=False, na=False))
          .any(axis=1)
    )
    opening_rows = (
        df[opening_mask]
          .replace(r"^\s*$", np.nan, regex=True)
          .dropna(axis=1, how="all")
    )

    if not opening_rows.empty and opening_rows.shape[1] > 1:
        raw_val = opening_rows.iloc[0, 1]   # column[1] holds the value
        dim_rec["opening_balance"] = pd.to_numeric(
            str(raw_val).replace(",", ""), errors="coerce"
        )
    else:
        dim_rec["opening_balance"] = None
    df = df.dropna(how='all')
    
    pattern = "opening|total|page"

    # df = df.dropna(axis=1, how='all')
    df = df[~df.apply(lambda x: x.astype(str).str.contains(pattern, case=False).any(), axis=1)]
    df = df.reset_index(drop=True)

    meta_keys = ["branch_code", "branch_name", "acc_no"]
    for key in meta_keys:
        df[key] = dim_rec.get(key)

    return df, dim_rec
    


In [ ]:
bank_df = []
dim_records = []
for file in Path(BANK_DIR).glob("*.xls"):
    df,dim_rec = process_bank_file(file)

    if df is not None:
        bank_df.append(df)

    if dim_rec is not None:
        dim_records.append(dim_rec)

bank_df = pd.concat(bank_df, ignore_index=True)
bank_bal = pd.DataFrame(dim_records)
bank_df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,source_dim_col,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,branch_code,branch_name,acc_no
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-17,NaN,10:06:31,0182,TIRUPUR LAKSHMI NAGAR,40168221385
1,Txn. Date,Cheque No.,NaN,Particulars,NaN,NaN,NaN,Credit Amount,NaN,Debit amount,NaN,Balance Amount,NaN,NaN,NaT,Value Date,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
2,2026-03-02 00:00:00,NaN,NaN,CASH DEPOSIT-CASH DEPOSIT SELF--,NaN,NaN,NaN,1000000,NaN,0,NaN,1353857.6,NaN,NaN,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
3,2026-03-02 00:00:00,NaN,NaN,CHEQUE WDL-CHEQUE TRANSFER TO--189264,NaN,NaN,NaN,0,NaN,62950,NaN,1290907.6,NaN,NaN,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
4,2026-03-02 00:00:00,NaN,NaN,CHQ TRANSFER-RTGS UTR NO: SBINR520260302186053...,NaN,NaN,NaN,0,NaN,382075,NaN,908832.6,NaN,NaN,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385


In [ ]:
bank_df['branch_code'].unique()

<StringArray>
['0182', '0627', '1046', '4304']
Length: 4, dtype: str

In [ ]:
# Find the header row
mask = bank_df.astype(str).apply(
    lambda col: col.str.contains("date", case=False, na=False)
)

header_row_idx = mask.any(axis=1).idxmax()

df = bank_df.iloc[header_row_idx + 1:].copy()
df.columns = bank_df.iloc[header_row_idx]
df = df.reset_index(drop=True)

df.head()

1,Txn. Date,Cheque No.,NaN,Particulars,NaN,NaN,NaN,Credit Amount,NaN,Debit amount,NaN,Balance Amount,NaN,NaN,NaT,Value Date,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
0,2026-03-02 00:00:00,NaN,NaN,CASH DEPOSIT-CASH DEPOSIT SELF--,NaN,NaN,NaN,1000000,NaN,0,NaN,1353857.6,NaN,NaN,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
1,2026-03-02 00:00:00,NaN,NaN,CHEQUE WDL-CHEQUE TRANSFER TO--189264,NaN,NaN,NaN,0,NaN,62950,NaN,1290907.6,NaN,NaN,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
2,2026-03-02 00:00:00,NaN,NaN,CHQ TRANSFER-RTGS UTR NO: SBINR520260302186053...,NaN,NaN,NaN,0,NaN,382075,NaN,908832.6,NaN,NaN,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
3,2026-03-02 00:00:00,NaN,NaN,CHQ TRANSFER-NEFT UTR NO: SBIN526061130793--18...,NaN,NaN,NaN,0,NaN,35040,NaN,873792.6,NaN,NaN,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
4,2026-03-03 00:00:00,NaN,NaN,CHQ TRANSFER-RTGS UTR NO: SBINR520260303186846...,NaN,NaN,NaN,0,NaN,800047.2,NaN,73745.4,NaN,NaN,NaT,2026-03-03 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385


In [ ]:

df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.dropna(axis=1, how='all')
df

1,Txn. Date,Cheque No.,Particulars,Credit Amount,Debit amount,Balance Amount,NaT,Value Date,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
0,2026-03-02 00:00:00,NaN,CASH DEPOSIT-CASH DEPOSIT SELF--,1000000,0,1353857.6,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
1,2026-03-02 00:00:00,NaN,CHEQUE WDL-CHEQUE TRANSFER TO--189264,0,62950,1290907.6,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
2,2026-03-02 00:00:00,NaN,CHQ TRANSFER-RTGS UTR NO: SBINR520260302186053...,0,382075,908832.6,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
3,2026-03-02 00:00:00,NaN,CHQ TRANSFER-NEFT UTR NO: SBIN526061130793--18...,0,35040,873792.6,NaT,2026-03-02 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
4,2026-03-03 00:00:00,NaN,CHQ TRANSFER-RTGS UTR NO: SBINR520260303186846...,0,800047.2,73745.4,NaT,2026-03-03 00:00:00,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385
...,...,...,...,...,...,...,...,...,...,...,...,...
260,2026-03-30 00:00:00,NaN,CASH DEPOSIT-CASH DEPOSIT SELF--,1965000,0,2148237.39,NaT,2026-03-30 00:00:00,NaN,4304,GAJWEL (AP),33069117110
261,2026-03-30 00:00:00,NaN,CASH HANDLING CHARGES---38976288,0,1739.03,2146498.36,NaT,2026-03-30 00:00:00,NaN,4304,GAJWEL (AP),33069117110
262,2026-03-30 00:00:00,NaN,CHEQUE WDL-CHEQUE TRANSFER TO--177185,0,1800000,346498.36,NaT,2026-03-30 00:00:00,NaN,4304,GAJWEL (AP),33069117110
263,2026-03-31 00:00:00,NaN,CASH DEPOSIT-CASH DEPOSIT SELF--,780000,0,1126498.36,NaT,2026-03-31 00:00:00,NaN,4304,GAJWEL (AP),33069117110


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Txn. Date                262 non-null    object        
 1   Cheque No.               3 non-null      str           
 2   Particulars              262 non-null    str           
 3   Credit Amount            262 non-null    object        
 4   Debit amount             262 non-null    object        
 5   Balance Amount           262 non-null    object        
 6   NaT                      3 non-null      datetime64[us]
 7   Value Date               262 non-null    object        
 8   nan                      3 non-null      object        
 9   0182                     265 non-null    str           
 10  TIRUPUR   LAKSHMI NAGAR  265 non-null    str           
 11  40168221385              265 non-null    str           
dtypes: datetime64[us](1), object(6), str(5)
memory 

In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)


cols = df.columns.tolist()

cols[0] = "txn_date"
cols[9] = "branch_code"
cols[10] = "branch_name"
cols[11] = "acc_no"
df.columns = cols

# Convert date column once, outside the loop
date_cols = ["txn_date", "value_date"]
for date_col in date_cols:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

# Convert numeric columns
cols = ['credit_amount', 'debit_amount', 'balance_amount']
for col in cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    ).fillna(0)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   txn_date        259 non-null    datetime64[us]
 1   cheque_no.      3 non-null      str           
 2   particulars     262 non-null    str           
 3   credit_amount   265 non-null    float64       
 4   debit_amount    265 non-null    float64       
 5   balance_amount  265 non-null    float64       
 6   NaT             3 non-null      datetime64[us]
 7   value_date      259 non-null    datetime64[us]
 8   nan             3 non-null      object        
 9   branch_code     265 non-null    str           
 10  branch_name     265 non-null    str           
 11  acc_no          265 non-null    str           
dtypes: datetime64[us](3), float64(3), object(1), str(5)
memory usage: 25.0+ KB


In [ ]:
df["txn_type"] = np.select(
    [
        df["credit_amount"] > 0,
        df["debit_amount"] > 0
    ],
    ["cr", "dr"],
    default=None
)

In [ ]:
df[["cheque_no", "cust_name"]] = (
    df["particulars"]
    .astype(str)
    .str.extract(r"--(\d{6})\s+(.+)$")
)
df

,txn_date,cheque_no.,particulars,credit_amount,debit_amount,balance_amount,NaT,value_date,NaN,branch_code,branch_name,acc_no,txn_type,cheque_no,cust_name
0,2026-03-02,NaN,CASH DEPOSIT-CASH DEPOSIT SELF--,1000000.0,0.00,1353857.60,NaT,2026-03-02,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,cr,NaN,NaN
1,2026-03-02,NaN,CHEQUE WDL-CHEQUE TRANSFER TO--189264,0.0,62950.00,1290907.60,NaT,2026-03-02,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,dr,NaN,NaN
2,2026-03-02,NaN,CHQ TRANSFER-RTGS UTR NO: SBINR520260302186053...,0.0,382075.00,908832.60,NaT,2026-03-02,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,dr,189263,MAHALAKSHMI
3,2026-03-02,NaN,CHQ TRANSFER-NEFT UTR NO: SBIN526061130793--18...,0.0,35040.00,873792.60,NaT,2026-03-02,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,dr,189265,THANGAPANDIAN
4,2026-03-03,NaN,CHQ TRANSFER-RTGS UTR NO: SBINR520260303186846...,0.0,800047.20,73745.40,NaT,2026-03-03,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,dr,189266,MUTHOOT FINANCE LTD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260,2026-03-30,NaN,CASH DEPOSIT-CASH DEPOSIT SELF--,1965000.0,0.00,2148237.39,NaT,2026-03-30,NaN,4304,GAJWEL (AP),33069117110,cr,NaN,NaN
261,2026-03-30,NaN,CASH HANDLING CHARGES---38976288,0.0,1739.03,2146498.36,NaT,2026-03-30,NaN,4304,GAJWEL (AP),33069117110,dr,NaN,NaN
262,2026-03-30,NaN,CHEQUE WDL-CHEQUE TRANSFER TO--177185,0.0,1800000.00,346498.36,NaT,2026-03-30,NaN,4304,GAJWEL (AP),33069117110,dr,NaN,NaN
263,2026-03-31,NaN,CASH DEPOSIT-CASH DEPOSIT SELF--,780000.0,0.00,1126498.36,NaT,2026-03-31,NaN,4304,GAJWEL (AP),33069117110,cr,NaN,NaN


In [ ]:
df['branch_code'].unique()

<StringArray>
['0182', '0627', '1046', '4304']
Length: 4, dtype: str

In [1]:
df.head()

NameError: name 'df' is not defined

In [23]:
REQUIRED_COLUMNS = [
    "txn_date", "cheque_no", "particulars", "cust_name", "credit_amount", "debit_amount", "balance_amount", "value_date", "branch_code", "branch_name", "acc_no", "txn_type"
]

In [24]:
df.columns

Index([      'txn_date',     'cheque_no.',    'particulars',  'credit_amount',
         'debit_amount', 'balance_amount',              NaT,     'value_date',
                    nan,    'branch_code',    'branch_name',         'acc_no',
             'txn_type',      'cheque_no',      'cust_name'],
      dtype='object')

In [29]:
available_cols = [c for c in REQUIRED_COLUMNS if c in df.columns]
df = df[available_cols]
missing = set(REQUIRED_COLUMNS) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")


In [28]:
df.head()

,txn_date,cheque_no.,particulars,credit_amount,debit_amount,balance_amount,NaT,value_date,NaN,branch_code,branch_name,acc_no,txn_type,cheque_no,cust_name
0,2026-03-02,NaN,CASH DEPOSIT-CASH DEPOSIT SELF--,1000000.0,0.0,1353857.6,NaT,2026-03-02,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,cr,NaN,NaN
1,2026-03-02,NaN,CHEQUE WDL-CHEQUE TRANSFER TO--189264,0.0,62950.0,1290907.6,NaT,2026-03-02,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,dr,NaN,NaN
2,2026-03-02,NaN,CHQ TRANSFER-RTGS UTR NO: SBINR520260302186053...,0.0,382075.0,908832.6,NaT,2026-03-02,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,dr,189263,MAHALAKSHMI
3,2026-03-02,NaN,CHQ TRANSFER-NEFT UTR NO: SBIN526061130793--18...,0.0,35040.0,873792.6,NaT,2026-03-02,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,dr,189265,THANGAPANDIAN
4,2026-03-03,NaN,CHQ TRANSFER-RTGS UTR NO: SBINR520260303186846...,0.0,800047.2,73745.4,NaT,2026-03-03,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,dr,189266,MUTHOOT FINANCE LTD


In [34]:
# df.to_csv("cleaned_bank_fact.csv", index=False)

In [35]:
# Extract last balance for each account number
last_balance = (
    df.groupby("acc_no")["balance_amount"]
    .last()
    .reset_index()
    .rename(columns={"balance_amount": "closing_balance"})
)

# Merge into dim_balance
bank_bal = bank_bal.merge(last_balance, on="acc_no", how="left")

bank_bal['label'] = "Bank"
bank_bal

,branch_code,branch_name,acc_no,opening_balance,closing_balance,label
0,0182,TIRUPUR LAKSHMI NAGAR,40168221385,353857.60,2162209.00,Bank
1,0627,HYDERABAD VANASTALIPURAM,43143793734,2759866.14,7749226.14,Bank
2,1046,UTHUKOTTAI,43782239368,2092603.80,344678.80,Bank
3,4304,GAJWEL (AP),33069117110,400014.04,1125808.06,Bank


In [1]:
#--- Database connection test ---
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "host":     os.getenv("DB_HOST"),
        "port":     os.getenv("DB_PORT"),
        "dbname":   os.getenv("DB_NAME"),
        "user":     os.getenv("DB_USER"),
        "password": os.getenv("DB_PASSWORD"),
    }
)

# Test the connection
with engine.connect() as conn:
    print("✅ Connected to PostgreSQL successfully!")

✅ Connected to PostgreSQL successfully!


In [12]:
import pandas as pd
final_bank_df = pd.read_csv("C:/Users/user/Documents/NESA/Muthoot/Branch BRS Automation/Final Data/cleaned_bank_fact 1.csv",  encoding="latin1")
final_bank_df.head()

,txn_seq,txn_date,cheque_no,particulars,cust_name,credit_amount,debit_amount,balance_amount,value_date,branch_code,branch_name,acc_no,txn_type
0,1,02-03-2026,NaN,CASH DEPOSIT-CASH DEPOSIT SELF--,NaN,1000000,0.0,1353857.6,02-03-2026,182,TIRUPUR LAKSHMI NAGAR,40168221385,cr
1,2,02-03-2026,189264.0,CHEQUE WDL-CHEQUE TRANSFER TO--189264,NaN,0,62950.0,1290907.6,02-03-2026,182,TIRUPUR LAKSHMI NAGAR,40168221385,dr
2,3,02-03-2026,189263.0,CHQ TRANSFER-RTGS UTR NO: SBINR520260302186053...,MAHALAKSHMI,0,382075.0,908832.6,02-03-2026,182,TIRUPUR LAKSHMI NAGAR,40168221385,dr
3,4,02-03-2026,189265.0,CHQ TRANSFER-NEFT UTR NO: SBIN526061130793--18...,THANGAPANDIAN,0,35040.0,873792.6,02-03-2026,182,TIRUPUR LAKSHMI NAGAR,40168221385,dr
4,5,03-03-2026,189266.0,CHQ TRANSFER-RTGS UTR NO: SBINR520260303186846...,MUTHOOT FINANCE LTD,0,800047.2,73745.4,03-03-2026,182,TIRUPUR LAKSHMI NAGAR,40168221385,dr


In [14]:
len(final_bank_df.columns)

13

In [13]:
#--- Insert data into PostgreSQL ---
final_bank_df.to_sql(
    name = "bank_fact",
    con = engine,
    if_exists = "replace",
    index = False,
    chunksize = 1000
)

print(f"✅ {len(final_bank_df)} rows inserted successfully!")

✅ 259 rows inserted successfully!


In [107]:
# Insert data into PostgreSQL
bank_bal.to_sql(
    name = "bank_balance",
    con = engine,
    if_exists = "replace",
    index = False,
    method="multi"
)

print(f"✅ {len(bank_bal)} rows inserted successfully!")

✅ 4 rows inserted successfully!


In [108]:
result = pd.read_sql("SELECT * FROM bank_balance LIMIT 5;", con=engine)
print(result)

  branch_code                branch_name       acc_no  opening_balance  \
0        0182    TIRUPUR   LAKSHMI NAGAR  40168221385        353857.60   
1        0627  HYDERABAD  VANASTALIPURAM  43143793734       2759866.14   
2        1046                 UTHUKOTTAI  43782239368       2092603.80   
3        4304              GAJWEL   (AP)  33069117110        400014.04   

   closing_balance label  
0       2162209.00  Bank  
1       7749226.14  Bank  
2        344678.80  Bank  
3       1125808.06  Bank  
